#01 Perforamance Tuning

##1. Application Code Level Optimization

### Cache, Use REduceByKEy instead of groupBYKey

##2. Cluster Level Optimization

### Containers and Executors

#✅ 01. Performance Tuning in Apache Spark
Performance tuning in Spark is divided into two major areas:

| Category                  | Focus Area                       | Goal                                                  |
| ------------------------- | -------------------------------- | ----------------------------------------------------- |
| 🔹 Application Code Level | Optimize logic and operations    | Reduce shuffles, memory usage, and execution time     |
| 🔹 Cluster Level          | Tune Spark cluster configuration | Use cluster resources efficiently (CPU, memory, etc.) |



#🔹 1. Application Code Level Optimization

| # | Technique                                     | Explanation                                                                                         | Example                                                                                           | Benefit                               |
| - | --------------------------------------------- | --------------------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------- | ------------------------------------- |
| 1 | **Use `reduceByKey` instead of `groupByKey`** | `groupByKey` shuffles all values over the network; `reduceByKey` aggregates locally before shuffle. | `rdd.groupByKey().mapValues(sum)` ❌<br>`rdd.reduceByKey(lambda a,b: a+b)` ✅                       | Less data shuffled = faster execution |
| 2 | **Use `cache()` or `persist()`**              | If you're reusing a DataFrame/RDD, cache it to avoid recomputation                                  | `df.cache().count()`<br>`df.cache().show()`                                                       | Faster second-time access             |
| 3 | **Avoid using `collect()` on large datasets** | `collect()` brings data to driver. Use `take()`, `show()`, or write to disk instead.                | `df.collect()` ❌<br>`df.show(10)` ✅                                                               | Avoids driver crash due to OOM        |
| 4 | **Use narrow transformations**                | Prefer narrow transformations like `map`, `filter` which don’t shuffle data                         | `rdd.filter(lambda x: x > 10).map(...)`                                                           | Reduces network overhead              |
| 5 | **Use broadcast joins for small tables**      | When joining large and small datasets, broadcast the small one                                      | `broadcast_df = spark.broadcast(small_df)`<br>`large_df.join(broadcast(small_df), "key")`         | Reduces shuffle join                  |
| 6 | **Avoid unnecessary columns**                 | Select only needed columns using `select()` or projection                                           | `df.select("id", "name")`                                                                         | Reduces memory & CPU usage            |
| 7 | **Avoid UDFs when possible**                  | Use Spark SQL built-in functions instead of Python UDFs                                             | ❌ `df.withColumn("upper", my_udf(col("name")))`<br>✅ `df.withColumn("upper", upper(col("name")))` | UDFs break Catalyst optimization      |
| 8 | **Filter early (predicate pushdown)**         | Push filters as early as possible                                                                   | `df.filter("date >= '2024-01-01'")` before join                                                   | Reduces input size for processing     |


#🔹 2. Cluster Level Optimization
| # | Component           | Tuning Parameter                  | Explanation                                 | Example / Value                              | Benefit                                 |
| - | ------------------- | --------------------------------- | ------------------------------------------- | -------------------------------------------- | --------------------------------------- |
| 1 | Executor Memory     | `--executor-memory`               | Amount of RAM per executor                  | `--executor-memory 4G`                       | Prevents OOM / Efficient memory         |
| 2 | Executor Cores      | `--executor-cores`                | Number of tasks per executor                | `--executor-cores 4`                         | Better parallelism                      |
| 3 | Number of Executors | `--num-executors`                 | Total number of executors                   | `--num-executors 10`                         | Utilizes full cluster                   |
| 4 | Driver Memory       | `--driver-memory`                 | Memory for Spark driver                     | `--driver-memory 2G`                         | Helps handle metadata and small results |
| 5 | Dynamic Allocation  | `spark.dynamicAllocation.enabled` | Adjusts executors dynamically based on load | `true`                                       | Saves resources                         |
| 6 | Shuffle Partitions  | `spark.sql.shuffle.partitions`    | Number of partitions during shuffles        | Default is 200, reduce if data is small      | Balances stage performance              |
| 7 | GC Settings         | JVM GC configs                    | Tune for large heap sizes                   | `-XX:+UseG1GC`                               | Reduces GC overhead                     |
| 8 | Locality            | `spark.locality.wait`             | Controls wait time for locality             | `spark.locality.wait=3s`                     | Avoids stragglers and delays            |
| 9 | Serialization       | `spark.serializer`                | Use Kryo for efficient serialization        | `org.apache.spark.serializer.KryoSerializer` | Faster and compact serialization        |


 Example Spark Submit Tuning Command

spark-submit \
  --executor-memory 4G \
  --executor-cores 4 \
  --num-executors 10 \
  --driver-memory 2G \
  --conf "spark.sql.shuffle.partitions=100" \
  --conf "spark.serializer=org.apache.spark.serializer.KryoSerializer" \
  app.py


#✅ Summary Table
| Aspect             | Poor Practice          | Optimized Approach                |
| ------------------ | ---------------------- | --------------------------------- |
| Key Aggregation    | `groupByKey()`         | `reduceByKey()`                   |
| Repeated Data Use  | Recompute              | `cache()` or `persist()`          |
| Join Strategy      | Shuffle join           | Broadcast small table             |
| Data Load          | Load all columns       | Use `select()`                    |
| Serialization      | Java serialization     | Kryo serialization                |
| Executor Setup     | Defaults               | Proper tuning for memory & cores  |
| Shuffle Partitions | Too many (200 default) | Tune based on data size           |
| Collecting Results | `collect()` large data | `take()`, `show()`, write to file |
